# Features and transparent rule baseline

Stage 2 builds one feature row per candidate at T0 and again at T1, then
applies a **recency-vote rule**. The rule is the auditable baseline, not the
final model. Matching rules and the score formula are in `docs/DATA_FLOW.md`.


## What the score is

Each source votes on whether Delaware or out-of-state evidence is **newer**.
Votes are weighted (title and address 2.0, license 1.5, work 1.0, external 0.5)
and a current DE/OOS address adds ±1.5.

- score ≥ 2.0 → `review_warranted`
- score ≤ -2.0 → `review_not_warranted`
- thin file or mixed score → `insufficient_evidence`

Thresholds come from the weights (one strong DE-newer source is enough), not
from a search on the 300 labels.


In [1]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "oos_review").exists() else Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from oos_review.evaluate import summarize
from oos_review.load import load_labels
from oos_review.pipeline import run_features_and_baseline

pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 140)

features, preds = run_features_and_baseline(save=True)
labels = load_labels()
print(features.shape, "feature rows;", preds.shape, "prediction rows")
print("T0 class mix:", preds.loc[preds.phase.eq("T0"), "predicted_class"].value_counts().to_dict())
print("T1 class mix:", preds.loc[preds.phase.eq("T1"), "predicted_class"].value_counts().to_dict())


(24000, 51) feature rows; (24000, 57) prediction rows
T0 class mix: {'review_not_warranted': 4253, 'review_warranted': 4171, 'insufficient_evidence': 3576}
T1 class mix: {'review_warranted': 4231, 'review_not_warranted': 3927, 'insufficient_evidence': 3842}


## Score separates the labeled classes

On the development set the mean score sits in the intended order: warranted
positive, insufficient near zero, not-warranted negative. Accuracy is well
above chance (~0.33) and well below a fitted model — that is the point of a
baseline.


In [2]:
labeled = preds.merge(labels, on="candidate_record_id")
t0 = labeled.loc[labeled.phase.eq("T0")]
t1 = labeled.loc[labeled.phase.eq("T1")]

print("T0 mean de_oos_score by true label")
print(t0.groupby("label_t0")["de_oos_score"].agg(["mean", "median", "count"]).round(3).to_string())
print()
print("T1 mean de_oos_score by true label")
print(t1.groupby("label_t1")["de_oos_score"].agg(["mean", "median", "count"]).round(3).to_string())

s0 = summarize(t0["label_t0"], t0["predicted_class"])
s1 = summarize(t1["label_t1"], t1["predicted_class"])
print(f"\nT0 accuracy={s0['accuracy']:.3f} macro-F1={s0['macro_f1']:.3f}")
print(s0["per_class"].round(3).to_string(index=False))
print(s0["confusion"].to_string())
print(f"\nT1 accuracy={s1['accuracy']:.3f} macro-F1={s1['macro_f1']:.3f}")
print(s1["per_class"].round(3).to_string(index=False))
print(s1["confusion"].to_string())


T0 mean de_oos_score by true label
                        mean  median  count
label_t0                                   
insufficient_evidence -0.071    0.00    105
review_not_warranted  -2.267   -3.00    105
review_warranted       2.211    1.75     90

T1 mean de_oos_score by true label
                        mean  median  count
label_t1                                   
insufficient_evidence  0.019     0.5    103
review_not_warranted  -1.454    -1.5    108
review_warranted       2.258     2.5     89

T0 accuracy=0.470 macro-F1=0.471
                class  precision  recall    f1  support
     review_warranted      0.479   0.500 0.489       90
 review_not_warranted      0.574   0.552 0.563      105
insufficient_evidence      0.362   0.362 0.362      105
pred                   review_warranted  review_not_warranted  insufficient_evidence
true                                                                                
review_warranted                     45                    10

## T1 must be allowed to change the call

The T1 stream is later evidence. If T0 and T1 predictions were identical, the
rule would be ignoring it.


In [3]:
wide = preds.pivot(index="candidate_record_id", columns="phase", values="predicted_class")
print("share of cases whose class changes T0→T1:", float((wide["T0"] != wide["T1"]).mean()))
print()
print(pd.crosstab(wide["T0"], wide["T1"], margins=True).to_string())


share of cases whose class changes T0→T1: 0.30825

T1                     insufficient_evidence  review_not_warranted  review_warranted    All
T0                                                                                         
insufficient_evidence                   2251                   589               736   3576
review_not_warranted                     822                  2993               438   4253
review_warranted                         769                   345              3057   4171
All                                     3842                  3927              4231  12000


## Example reasons

`rule_reason` is the audit line a reviewer should see next to the class.


In [4]:
examples = [
    ("CAN-2B1096MPJS", "labeled review_warranted at T0 and T1"),
    ("CAN-1EJVSHCWNY", "labeled review_not_warranted at T0 and T1"),
    ("CAN-QCF4GZN8TD", "labeled insufficient_evidence at T0 and T1"),
    ("CAN-O523BGV6IR", "labeled insufficient → not_warranted"),
]
show_cols = [
    "phase", "predicted_class", "de_oos_score",
    "p_review_warranted", "p_review_not_warranted", "p_insufficient_evidence",
    "review_priority", "rule_reason",
]
for cid, note in examples:
    print("=" * 72)
    print(cid, "—", note)
    print(labels.loc[labels.candidate_record_id.eq(cid)].to_string(index=False))
    print(preds.loc[preds.candidate_record_id.eq(cid), show_cols].to_string(index=False))
    print()


CAN-2B1096MPJS — labeled review_warranted at T0 and T1
candidate_record_id         label_t0         label_t1
     CAN-2B1096MPJS review_warranted review_warranted
phase  predicted_class  de_oos_score  p_review_warranted  p_review_not_warranted  p_insufficient_evidence  review_priority                                                                                                                                                                                 rule_reason
   T0 review_warranted           2.5             0.77096                0.044278                 0.184761         0.624411 address DE-newer (+2.0); license DE-newer (+1.5); title OOS-newer (-2.0); work OOS-newer (-1.0); external DE-newer (+0.5); current address DE (+1.5); score=+2.5; decision=review_warranted
   T1 review_warranted           2.5             0.77096                0.044278                 0.184761         0.624411 address DE-newer (+2.0); license DE-newer (+1.5); title OOS-newer (-2.0); work OOS-newer (-1

## Next

Keep `de_oos_score` and the recency votes as features. Fit a 3-class model
with nested CV on the 300 labels, and compare it to this rule on accuracy,
calibration, and whether T1 actually moves the prediction.
